# Versuch v7–v13: Quellen mit Zitatprüfung

Ziel: Falsche Quellenangaben automatisch erkennen, statt sie von Hand zu suchen.

- Jeder Bericht bekommt eine Kennung (`B1`, `B2`, ...). Das Modell schreibt keine Quellenliste mehr, sondern belegt **jeden Satz** mit Kennung und kurzem **wörtlichem Zitat**.
- Das Notebook prüft, ob das Zitat wirklich im genannten Bericht steht. Nur geprüfte Belege werden als Quelle angezeigt; **Titel und Datum kommen aus den Berichten selbst**, nicht vom Modell.
- Aufbau sonst wie v5/v6: alle Berichte vollständig im Prompt, keine Suche, mehrere Durchläufe.
- **v8:** Einzige Änderung gegenüber v7 ist Regel 7 (knappe Antworten, keine Verweise wie „ein Bericht“ im Text, weil die Fußnoten die Zuordnung zeigen).
- **v9:** Einzige Änderung gegenüber v8 sind vier Berichte mit abweichendem Format (Datum und Name an anderer Stelle oder fehlend, teils ohne Überschrift; Inhalt gleich). Titel und Datum liest dafür ein erweitertes Programm aus.
- **v10:** Kurzer Datumstest. Zwei neue Berichte zur Schlüsselrückgabe widersprechen sich ohne sprachlichen Hinweis auf die Reihenfolge; der neuere steht zuerst. Nur Fragen 18 und 19 (`NUR_FRAGEN`).
- **v11:** Vergleichsbasis für die Suche. Alle 48 Berichte (`dataset-v3`) im Prompt, alle 25 Fragen, sonst wie v8 (`K = None`).
- **v12:** Einzige Änderung gegenüber v11: Das Modell bekommt je Frage nur die **5 besten Treffer der Suche nach Bedeutung** (`intfloat/multilingual-e5-small`, wie im Suchtest s1), nicht alle Berichte (`K = 5`). Kennungen B1, B2, ... bleiben dieselben wie in v11.
- **v13:** Gründliche Prüfung statt Suche. Einzige Änderung gegenüber v12: Das Modell **sichtet alle Berichte in Paketen zu je 10** und gibt je Paket die Kennungen der relevanten Berichte zurück; nur diese gehen in den Antwortschritt (Regeln und Zitatprüfung unverändert). Die Pakete simulieren eine Sammlung, die nicht in eine einzige Anfrage passt. Gilt für alle Fragen (`SICHTEN = True`). Kosten je Anfrage werden mitgespeichert, soweit OpenRouter sie meldet.
- **Grenze:** Ein gefundenes Zitat zeigt, dass der Bericht den Satz enthält, nicht dass die Aussage der Antwort daraus folgt.

In [ ]:
!pip install -q openai sentence-transformers
!git clone -q https://github.com/Julien0049/AI-database.git || git -C AI-database pull -q

In [ ]:
from google.colab import userdata
from openai import OpenAI

KEY_NAME = "OPENROUTER_API_KEY"
MODEL = "google/gemini-2.5-pro"

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get(KEY_NAME))

## Berichte mit Kennung und Metadaten

Titel und Datum liest das Programm aus, nicht das Modell (ab v9):

- **Titel:** erste Markdown-Überschrift (`# ...`); fehlt sie, steht der Dateiname in der Fußnote.
- **Datum:** frühestes Datum im Text in einem der Formate `15.2.2023`, `2025-01-20` oder `18. März 2025`, egal an welcher Stelle; fehlt es, steht „Datum unbekannt“.
- **Grenze:** Andere Formate werden nicht erkannt. Steht im Text vor dem Berichtsdatum ein anderes Datum, wird dieses genommen.

In [ ]:
import pathlib, re

DATA = pathlib.Path("AI-database/data/synthetic_reports")

MONTHS = {m: i for i, m in enumerate(["januar", "februar", "märz", "april", "mai", "juni", "juli",
                                      "august", "september", "oktober", "november", "dezember"], 1)}
DATE_PATTERNS = [
    (r"\b(\d{1,2})\.(\d{1,2})\.(\d{4})\b", lambda m: (m[1], m[2], m[3])),                          # 15.2.2023
    (r"\b(\d{4})-(\d{2})-(\d{2})\b", lambda m: (m[3], m[2], m[1])),                                # 2025-01-20
    (r"\b(\d{1,2})\. (" + "|".join(MONTHS) + r") (\d{4})\b", lambda m: (m[1], MONTHS[m[2]], m[3])),  # 18. März 2025
]

def find_title(text):
    """Erste Markdown-Überschrift; None, wenn keine vorhanden ist."""
    m = re.search(r"^# (?:After-Action-Report: )?(.+)$", text, re.M)
    return m.group(1).strip() if m else None

def find_date(text):
    """Frühestes Datum im Text, gleich in welchem der bekannten Formate; None, wenn keines vorkommt."""
    hits = []
    for pattern, parts in DATE_PATTERNS:
        for m in re.finditer(pattern, text, re.I):
            d, mo, y = parts([None] + [g.lower() for g in m.groups()])
            hits.append((m.start(), f"{int(d):02d}.{int(mo):02d}.{y}"))
    return min(hits)[1] if hits else None

reports = {}
for n, path in enumerate(sorted(DATA.glob("AAR-*.md")), 1):
    text = path.read_text()
    reports[f"B{n}"] = {"datei": path.name, "titel": find_title(text) or path.name,
                        "datum": find_date(text) or "Datum unbekannt", "text": text}

questions = re.findall(r"\*\*Frage\*\*\s*\n\s*> (.+)", (DATA / "test_questions.md").read_text())
print(len(reports), "Berichte,", len(questions), "Fragen")
for rid, r in reports.items():
    print(rid, r["datum"], r["titel"])

## Auswahl der Berichte: Suche (v12) oder Sichtung (ab v13)

`K = None`: alle Berichte im Prompt (bis v11). `K = 5`: je Frage nur die 5 Berichte, die der Frage nach Bedeutung am ähnlichsten sind; Verfahren wie im Suchtest s1 (ganzer Bericht als Einheit, Präfixe „query: “/„passage: “). Das Embedding-Modell läuft in Colab ohne GPU; der erste Aufruf lädt es herunter (ca. 470 MB).

`SICHTEN = True` (ab v13): Statt der Suche prüft das Modell alle Berichte in Paketen zu je `PAKET` Berichten darauf, ob sie etwas zur Frage beitragen, auch nebenbei. `K` wird dann nicht verwendet. Ist eine Sichtungsantwort nicht auswertbar, bleibt das ganze Paket ausgewählt, damit nichts verloren geht.

In [ ]:
K = None  # None = alle Berichte im Prompt; sonst Anzahl der Treffer je Frage
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
SICHTEN = True  # ab v13: Modell sichtet alle Berichte in Paketen; K wird dann nicht verwendet
PAKET = 10      # Berichte je Sichtungsanfrage

if K and not SICHTEN:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    ids = list(reports)
    vectors = embedder.encode([f"passage: {reports[rid]['text']}" for rid in ids], normalize_embeddings=True)

def select(question):
    """Kennungen der Berichte, die das Modell zu dieser Frage bekommt."""
    if not K:
        return list(reports)
    q = embedder.encode(f"query: {question}", normalize_embeddings=True)
    scores = vectors @ q
    return [ids[j] for j in sorted(range(len(ids)), key=lambda j: -scores[j])[:K]]

def corpus_for(selected):
    return "\n\n".join(f'<bericht id="{rid}">\n{reports[rid]["text"]}\n</bericht>' for rid in selected)

SICHTUNG = """Du prüfst fiktive Testberichte darauf, ob sie für eine Frage relevant sind. Jeder Bericht hat eine Kennung wie B1, B2.
Relevant ist ein Bericht, wenn er mindestens eine Aussage enthält, die zur Beantwortung der Frage beiträgt, auch wenn sie nur nebenbei erwähnt wird.
Nimm auch ältere oder einander widersprechende Stände auf; welcher Stand gilt, wird später entschieden.
Nicht relevant sind Berichte, die nur ähnliche Wörter in einem anderen Zusammenhang enthalten.
Antworte nur mit JSON in dieser Form: {"relevant": ["B3", "B7"]}. Wenn kein Bericht relevant ist: {"relevant": []}."""

## Antwortregeln und Ausgabeformat

Regeln 1–6 wie in v5/v6. Neu ist das Format: JSON mit Sätzen und Belegen. Ab v8 zusätzlich Regel 7.

In [ ]:
VERSION = "v13"

SYSTEM = """Du beantwortest Fragen ausschließlich auf Grundlage der mitgelieferten Berichte.
Die Berichte sind fiktive Testdaten. Jeder Bericht hat eine Kennung wie B1, B2.

Regeln:
1. Verwende nur Aussagen aus den Berichten. Ergänze nichts aus eigenem Wissen.
2. Ergänzende Aussagen aus mehreren Berichten darfst du zu einer klaren Antwort zusammenführen.
3. Bei Regeln und Verfahren: Wenn Berichte mit vergleichbarem Kontext sich widersprechen, bestimmt die jüngste passende Aussage die Antwort.
   Nenne ältere abweichende Verfahren nicht als gleichrangige Alternativen.
   Bei Erfahrungen und Einschätzungen (zum Beispiel, was sich bewährt hat): Gib unterschiedliche Sichtweisen nebeneinander wieder
   und weise darauf hin, dass die Berichte auseinandergehen. Das Datum allein begründet hier keine Priorität.
4. Wenn Berichte unterschiedliche Kontexte betreffen oder keine sichere Priorisierung zulassen, behaupte keine Priorität.
5. Bezeichne keine Quelle als gültig oder verbindlich; die Berichte belegen das nicht.
6. Wenn die Berichte die Frage nicht ausreichend beantworten, sag das offen und fülle keine Lücken.
   Das gilt auch für jede Teilfrage: Wenn zu einem Teil der Frage keine Informationen vorliegen, schreibe ausdrücklich,
   dass die Berichte dazu nichts sagen. Gib dann nicht stattdessen eine verwandte Aussage als Antwort aus.
7. Antworte exakt und knapp: nur was zur Frage gehört, ohne Einleitung und ohne Wiederholungen.
   Schreibe nicht, aus welchem Bericht eine Aussage stammt (zum Beispiel "ein Bericht", "ein anderer Bericht"); die Zuordnung wird automatisch als Fußnote ergänzt.
   Hinweise, dass die Berichte auseinandergehen oder zu etwas nichts sagen, gehören weiterhin in die Antwort.

Format: Antworte nur mit JSON in dieser Form:
{"saetze": [{"text": "Ein Satz der Antwort.", "belege": [{"bericht": "B3", "zitat": "wörtliche Stelle aus B3"}]}]}
- Jeder Satz, der eine Aussage aus den Berichten enthält, bekommt mindestens einen Beleg.
- Das Zitat ist eine kurze, wörtlich kopierte Stelle aus genau dem genannten Bericht (einige Wörter bis ein Satz).
- Nenne nur Berichte, aus denen die Aussage des Satzes stammt.
- Sätze, die eine Lücke benennen (die Berichte sagen dazu nichts), haben eine leere Beleg-Liste.
- Nenne in den Sätzen keine Titel, Daten oder Kennungen; die Quellen werden automatisch ergänzt."""

## Zitatprüfung

Vergleicht Zitate unabhängig von Groß-/Kleinschreibung, Leerzeichen, Anführungszeichen und Satzzeichen am Rand.

- **gültig:** Zitat steht im genannten Bericht.
- **falscher Bericht:** Zitat steht in einem anderen Bericht.
- **nicht gefunden:** Zitat steht in keinem Bericht (umformuliert oder erfunden).

In [ ]:
import json

def norm(s):
    s = s.lower()
    for a, b in [("„", '"'), ("“", '"'), ("”", '"'), ("‚", "'"), ("‘", "'"), ("’", "'"), ("–", "-"), ("—", "-")]:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip(" .,;:\"'")

def parse(raw):
    raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())
    return json.loads(raw)

def check(answer):
    for satz in answer["saetze"]:
        for beleg in satz.get("belege", []):
            zitat = norm(beleg.get("zitat", ""))
            rid = beleg.get("bericht")
            if rid in reports and zitat and zitat in norm(reports[rid]["text"]):
                beleg["status"] = "gültig"
            elif zitat and any(zitat in norm(r["text"]) for r in reports.values()):
                beleg["status"] = "falscher Bericht"
            else:
                beleg["status"] = "nicht gefunden"
    return answer

def render(answer):
    """Antworttext mit Fußnoten; aufeinanderfolgende Sätze aus denselben Berichten teilen sich eine Fußnote."""
    numbers = {}
    for s in answer["saetze"]:
        for b in s.get("belege", []):
            if b["status"] == "gültig":
                numbers.setdefault(b["bericht"], len(numbers) + 1)
    def refs(s):
        return sorted({numbers[b["bericht"]] for b in s.get("belege", []) if b["status"] == "gültig"})
    sentences = answer["saetze"]
    parts = []
    for i, s in enumerate(sentences):
        text, r = s["text"].strip(), refs(s)
        if r and (i + 1 == len(sentences) or refs(sentences[i + 1]) != r):
            mark = " [" + ", ".join(map(str, r)) + "]"
            text = text[:-1] + mark + text[-1] if text[-1:] in ".!?" else text + mark
        parts.append(text)
    out = " ".join(parts)
    if numbers:
        out += "\n\nQuellen:\n" + "\n".join(f'[{n}] „{reports[rid]["titel"]}“ ({reports[rid]["datum"]})'
                                           for rid, n in numbers.items())
    problems = [f'{b["bericht"]}: {b["status"]} – "{b.get("zitat", "")}"'
                for s in sentences for b in s.get("belege", []) if b["status"] != "gültig"]
    if problems:
        out += "\n\nNicht bestätigte Belege:\n" + "\n".join(f"- {p}" for p in problems)
    return out

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
import openai

RUNS = 3      # Anzahl Durchläufe; die Antworten schwanken zwischen Läufen
PARALLEL = 5  # Anzahl gleichzeitiger Anfragen
NUR_FRAGEN = None  # None = alle Fragen; sonst nur diese Nummern

def ask(system, user, retries=5):
    """Gibt (Antwort, Fehler, Kosten in USD) zurück; wartet bei Überlastung und versucht es erneut."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user}
                ]
            )
            return response.choices[0].message.content, None, getattr(response.usage, "cost", None)
        except (openai.RateLimitError, openai.APIConnectionError, openai.InternalServerError) as e:
            if attempt == retries - 1:
                return None, str(e), None
            time.sleep(5 * 2 ** attempt)
        except openai.APIStatusError as e:  # z. B. Guthaben aufgebraucht: nicht wiederholen, Lauf nicht abbrechen
            return None, str(e), None

jobs = [(d, i, q) for d in range(1, RUNS + 1) for i, q in enumerate(questions, 1)
        if NUR_FRAGEN is None or i in NUR_FRAGEN]
ids = list(reports)
batches = [ids[j:j + PAKET] for j in range(0, len(ids), PAKET)]
selection, sichtung, costs = {}, {}, []
start = time.time()
with ThreadPoolExecutor(max_workers=PARALLEL) as pool:
    if SICHTEN:
        checks = [(d, i, q, b) for d, i, q in jobs for b in batches]
        checked = list(pool.map(lambda c: ask(SICHTUNG, f"{corpus_for(c[3])}\n\nFrage: {c[2]}"), checks))
        for (d, i, q, b), (raw, error, cost) in zip(checks, checked):
            try:
                found = [rid for rid in parse(raw)["relevant"] if rid in b]
            except (json.JSONDecodeError, KeyError, TypeError, AttributeError):
                found = list(b)  # nicht auswertbar: ganzes Paket behalten
            selection.setdefault((d, i), []).extend(found)
            sichtung.setdefault((d, i), []).append({"paket": b, "relevant": found, "roh": raw, "fehler": error})
            costs.append(cost)
    else:
        selection = {(d, i): select(q) for d, i, q in jobs}  # Suche ist deterministisch
    answers = list(pool.map(lambda job: ask(SYSTEM, f"{corpus_for(selection[job[:2]])}\n\nFrage: {job[2]}"), jobs))
DURATION = round(time.time() - start)
costs += [cost for _, _, cost in answers]
COST = round(sum(c for c in costs if c), 4) if any(costs) else None
print(f"{len(costs)} Anfragen in {DURATION} s, Kosten: {COST} USD\n")

results = []
for (durchlauf, i, question), (raw, error, cost) in zip(jobs, answers):
    entry = {"durchlauf": durchlauf, "nr": i, "frage": question,
             "berichte_im_prompt": selection[(durchlauf, i)], "sichtung": sichtung.get((durchlauf, i)), "roh": raw}
    if error:
        entry["fehler"] = f"Anfrage fehlgeschlagen: {error}"
        shown = entry["fehler"]
    else:
        try:
            entry["antwort"] = check(parse(raw))
            shown = render(entry["antwort"])
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            entry["fehler"] = f"Ausgabe nicht auswertbar: {e}"
            shown = entry["fehler"] + "\n" + raw
    results.append(entry)
    print(f"## Durchlauf {durchlauf}, Frage {i}\n{question}\nBerichte: {', '.join(entry['berichte_im_prompt'])}\n\n{shown}\n")

## Kennzahlen

In [ ]:
from collections import Counter

status = Counter(b["status"] for e in results if "antwort" in e
                 for s in e["antwort"]["saetze"] for b in s.get("belege", []))
unbelegt = sum(1 for e in results if "antwort" in e for s in e["antwort"]["saetze"] if not s.get("belege"))
summary = {
    "antworten": len(results),
    "nicht_auswertbar": sum("fehler" in e for e in results),
    "belege": dict(status),
    "anteil_gueltig": round(status["gültig"] / max(sum(status.values()), 1), 3),
    "saetze_ohne_beleg": unbelegt,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

## Ergebnisse speichern

Sätze ohne Beleg sind nicht automatisch Fehler: Lückenhinweise haben bewusst keinen Beleg. Sie werden bei der Auswertung von Hand geprüft.

In [ ]:
import datetime
from google.colab import files

run = {
    "versuch": f"{VERSION} Zitatprüfung",
    "modell": MODEL,
    "zeitpunkt": datetime.datetime.now().isoformat(timespec="seconds"),
    "durchlaeufe": RUNS,
    "suche": {"k": None if SICHTEN else K, "embedding_modell": EMBEDDING_MODEL if K and not SICHTEN else None,
              "sichten": SICHTEN, "paket": PAKET if SICHTEN else None},
    "kosten_usd": COST,
    "dauer_s": DURATION,
    "systemprompt": SYSTEM,
    "berichte": {rid: {k: r[k] for k in ("datei", "titel", "datum")} for rid, r in reports.items()},
    "kennzahlen": summary,
    "ergebnisse": results,
}
name = f"ergebnisse_{MODEL.replace('/', '_')}_{datetime.date.today()}_{VERSION}.json"
pathlib.Path(name).write_text(json.dumps(run, ensure_ascii=False, indent=2))
files.download(name)